## Data Loading

In [1]:
import pandas as pd

file_path = '/content/uber_trips_march_test.parquet'
df = pd.read_parquet(file_path)
df.head()

,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,tolls,tips,driver_pay,PUBorough,PUZone,DOBorough,DOZone,time_from_request_to_pickup,day_of_week,hour_bucket
0,2026-03-01 00:02:07,2026-03-01 00:06:24,2026-03-01 00:07:02,2026-03-01 00:12:16,90,249,0.52,314,0.0,0.00,19.30,Manhattan,Flatiron,Manhattan,West Village,257.0,Sunday,0
1,2026-03-01 00:16:40,2026-03-01 00:20:28,2026-03-01 00:21:28,2026-03-01 00:31:52,113,211,1.16,625,0.0,0.00,23.36,Manhattan,Greenwich Village North,Manhattan,SoHo,228.0,Sunday,0
2,2026-03-01 00:36:45,2026-03-01 00:38:16,2026-03-01 00:40:09,2026-03-01 00:48:45,211,113,1.25,516,0.0,9.33,26.19,Manhattan,SoHo,Manhattan,Greenwich Village North,91.0,Sunday,0
3,2026-03-01 00:48:44,2026-03-01 00:51:48,2026-03-01 00:52:18,2026-03-01 01:07:45,114,170,2.38,927,0.0,5.35,22.19,Manhattan,Greenwich Village South,Manhattan,Murray Hill,184.0,Sunday,0
4,2026-02-28 23:54:23,2026-02-28 23:59:48,2026-03-01 00:01:47,2026-03-01 00:11:19,173,70,1.01,573,0.0,0.00,7.99,Queens,North Corona,Queens,East Elmhurst,325.0,Saturday,23


## Average Time For Zone To Zone (Per Hour Per Day)

In [2]:
grouped_data = df.groupby(['PUZone', 'DOZone', 'hour_bucket', 'day_of_week'])
average_trip_time_by_route_time = grouped_data['trip_time'].mean().reset_index()

# Calculate the count of instances for each combination
trip_count_by_route_time = grouped_data.size().reset_index(name='trip_count')

# Merge the count into the average trip time DataFrame
average_trip_time_by_route_time = pd.merge(
    average_trip_time_by_route_time,
    trip_count_by_route_time,
    on=['PUZone', 'DOZone', 'hour_bucket', 'day_of_week'],
    how='left'
)

# Set trip_time to 0 where PUZone and DOZone are the same
average_trip_time_by_route_time.loc[average_trip_time_by_route_time['PUZone'] == average_trip_time_by_route_time['DOZone'], 'trip_time'] = 0

# Rename the 'trip_time' column to 'average_trip_time'
average_trip_time_by_route_time = average_trip_time_by_route_time.rename(columns={'trip_time': 'average_PU_to_DO_time'})

print(average_trip_time_by_route_time.head())
print(average_trip_time_by_route_time.shape)

                    PUZone                   DOZone  hour_bucket day_of_week  \
0  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0      Friday   
1  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0      Monday   
2  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0    Saturday   
3  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0      Sunday   
4  Allerton/Pelham Gardens  Allerton/Pelham Gardens            0    Thursday   

   average_PU_to_DO_time  trip_count  
0                    0.0           7  
1                    0.0          10  
2                    0.0           6  
3                    0.0           8  
4                    0.0           4  
(2503385, 6)


In [3]:
# Merge 'average_PU_to_DO_time' back into the original DataFrame 'df'
df = pd.merge(
    df,
    average_trip_time_by_route_time[['PUZone', 'DOZone', 'hour_bucket', 'day_of_week', 'average_PU_to_DO_time']],
    on=['PUZone', 'DOZone', 'hour_bucket', 'day_of_week'],
    how='left'
)

print(df.head())
print(df.shape)

# Check for missing values after the merge
print("\nMissing values after merge:")
print(df.isnull().sum())

# Print 5 random rows with specified columns
print("\n5 random rows with relevant columns:")
print(df[['PUZone', 'DOZone', 'day_of_week', 'hour_bucket', 'average_PU_to_DO_time']].sample(5))

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-03-01 00:02:07 2026-03-01 00:06:24 2026-03-01 00:07:02   
1 2026-03-01 00:16:40 2026-03-01 00:20:28 2026-03-01 00:21:28   
2 2026-03-01 00:36:45 2026-03-01 00:38:16 2026-03-01 00:40:09   
3 2026-03-01 00:48:44 2026-03-01 00:51:48 2026-03-01 00:52:18   
4 2026-02-28 23:54:23 2026-02-28 23:59:48 2026-03-01 00:01:47   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  \
0 2026-03-01 00:12:16            90           249        0.52        314   
1 2026-03-01 00:31:52           113           211        1.16        625   
2 2026-03-01 00:48:45           211           113        1.25        516   
3 2026-03-01 01:07:45           114           170        2.38        927   
4 2026-03-01 00:11:19           173            70        1.01        573   

   tolls  tips  driver_pay  PUBorough                   PUZone  DOBorough  \
0    0.0  0.00       19.30  Manhattan                 Flatiron  Manhattan  

## Average Earnings For Pickup Zone (Per Hour Per Day)

In [4]:
import numpy as np

# Create actual calendar date so we don't group across many weeks/months
df['pickup_date'] = df['pickup_datetime'].dt.date

# -----------------------------
# Stage 1: daily zone-hour level
# -----------------------------
daily_zone_hour = df.groupby(
    ['PUZone', 'pickup_date', 'day_of_week', 'hour_bucket']
).agg(
    earliest_pickup_time=('pickup_datetime', 'min'),
    latest_dropoff_time=('dropoff_datetime', 'max'),
    total_driver_pay=('driver_pay', 'sum'),
    total_tips=('tips', 'sum'),
    trip_count=('pickup_datetime', 'count')
).reset_index()

daily_zone_hour['total_earnings'] = (
    daily_zone_hour['total_driver_pay'] + daily_zone_hour['total_tips']
)

daily_zone_hour['time_span_hours'] = (
    daily_zone_hour['latest_dropoff_time'] - daily_zone_hour['earliest_pickup_time']
).dt.total_seconds() / 3600

# Avoid bad/inflated values from tiny spans
daily_zone_hour = daily_zone_hour[daily_zone_hour['time_span_hours'] >= 0.10].copy()

daily_zone_hour['PU_driver_pay_per_hour'] = (
    daily_zone_hour['total_driver_pay'] / daily_zone_hour['time_span_hours']
)

daily_zone_hour['PU_tips_per_hour'] = (
    daily_zone_hour['total_tips'] / daily_zone_hour['time_span_hours']
)

daily_zone_hour['PU_total_earnings_per_hour'] = (
    daily_zone_hour['total_earnings'] / daily_zone_hour['time_span_hours']
)

# ------------------------------------------------
# Stage 2: average across same zone/day/hour pattern
# ------------------------------------------------
grouped_earnings = daily_zone_hour.groupby(
    ['PUZone', 'day_of_week', 'hour_bucket']
).agg(
    PU_avg_market_pay_per_hour=('PU_driver_pay_per_hour', 'mean'),
    PU_avg_market_tips_per_hour=('PU_tips_per_hour', 'mean'),
    PU_avg_market_total_earnings_per_hour=('PU_total_earnings_per_hour', 'mean'),
    avg_trip_count=('trip_count', 'mean'),
    avg_market_total_earnings=('total_earnings', 'mean'),
    avg_time_span_hours=('time_span_hours', 'mean'),
    sample_days=('pickup_date', 'nunique')
).reset_index()

print(grouped_earnings.head())
print(grouped_earnings.shape)

                    PUZone day_of_week  hour_bucket  \
0  Allerton/Pelham Gardens      Friday            0   
1  Allerton/Pelham Gardens      Friday            1   
2  Allerton/Pelham Gardens      Friday            2   
3  Allerton/Pelham Gardens      Friday            3   
4  Allerton/Pelham Gardens      Friday            4   

   PU_avg_market_pay_per_hour  PU_avg_market_tips_per_hour  \
0                  217.872776                     3.577753   
1                   99.916023                     4.621146   
2                   71.044001                     0.000000   
3                   95.852887                     0.000000   
4                  159.512861                     3.919012   

   PU_avg_market_total_earnings_per_hour  avg_trip_count  \
0                             221.450529           19.75   
1                             104.537168            9.75   
2                              71.044001            3.75   
3                              95.852887            5.50

In [5]:
top_earnings_rows = grouped_earnings.sort_values(by='avg_market_total_earnings', ascending=False)
print(top_earnings_rows.head())

               PUZone day_of_week  hour_bucket  PU_avg_market_pay_per_hour  \
25295  Midtown Center    Thursday           21                 8498.832739   
25343  Midtown Center   Wednesday           21                 8405.414975   
25319  Midtown Center     Tuesday           21                 9244.592043   
25294  Midtown Center    Thursday           20                 7737.110804   
25318  Midtown Center     Tuesday           20                 7957.764685   

       PU_avg_market_tips_per_hour  PU_avg_market_total_earnings_per_hour  \
25295                   738.306109                            9237.138848   
25343                   837.324851                            9242.739826   
25319                   891.997357                           10136.589400   
25294                   686.203619                            8423.314424   
25318                   762.455071                            8720.219757   

       avg_trip_count  avg_market_total_earnings  avg_time_span_hour

In [6]:
import numpy as np

# How frequently rides occur.
grouped_earnings['PU_trip_density_per_hour'] = (
    grouped_earnings['avg_trip_count'] /
    grouped_earnings['avg_time_span_hours']
)

# Trip Quality:
grouped_earnings['PU_avg_earnings_per_trip'] = (
    grouped_earnings['avg_market_total_earnings'] /
    grouped_earnings['avg_trip_count']
)

# Higher tipping neighborhoods.
grouped_earnings['PU_tip_ratio_per_trip'] = (
    grouped_earnings['PU_avg_market_tips_per_hour'] /
    grouped_earnings['PU_avg_market_total_earnings_per_hour']
)

grouped_earnings['PU_zone_opportunity_score'] = (
    grouped_earnings['PU_avg_market_total_earnings_per_hour']
    * np.log1p(grouped_earnings['PU_trip_density_per_hour'])
)

print(grouped_earnings.sample(5))

                          PUZone day_of_week  hour_bucket  \
6584        Central Harlem North   Wednesday           12   
17509    Greenwich Village North    Saturday            4   
41008  Williamsburg (South Side)   Wednesday            9   
41598                   Woodside    Saturday           23   
27089     New Dorp/Midland Beach    Saturday           15   

       PU_avg_market_pay_per_hour  PU_avg_market_tips_per_hour  \
6584                  1216.547867                    52.483652   
17509                  571.377576                    30.097361   
41008                 1311.775546                    71.715423   
41598                  706.554523                    28.679433   
27089                  208.450213                     9.355655   

       PU_avg_market_total_earnings_per_hour  avg_trip_count  \
6584                             1269.031519      155.500000   
17509                             601.474937       29.500000   
41008                            1383.490970

**Reasoning**:
The subtask requires calculating the mean of two columns from `grouped_earnings` and then dividing them to find a conversion factor. This can be done in a single code cell.



In [7]:
avg_pu_opportunity_score = grouped_earnings['PU_zone_opportunity_score'].mean()
avg_market_earnings = grouped_earnings['avg_market_total_earnings'].mean()

conversion_factor = avg_pu_opportunity_score / avg_market_earnings

print(f"Average PU Zone Opportunity Score: {avg_pu_opportunity_score}")
print(f"Average Market Total Earnings: {avg_market_earnings}")
print(f"Conversion Factor (Opportunity Score Index Units per Dollar): {conversion_factor}")

Average PU Zone Opportunity Score: 2996.5262916616184
Average Market Total Earnings: 1323.5499132515363
Conversion Factor (Opportunity Score Index Units per Dollar): 2.2640070175367377


In [8]:
columns_to_merge = [
    'PUZone',
    'day_of_week',
    'hour_bucket',
    'PU_zone_opportunity_score',
    'PU_tip_ratio_per_trip',
    'PU_avg_earnings_per_trip',
    'PU_trip_density_per_hour',
    'PU_avg_market_total_earnings_per_hour'
]

df = pd.merge(
    df,
    grouped_earnings[columns_to_merge],
    on=['PUZone', 'day_of_week', 'hour_bucket'],
    how='left'
)

print(df.head())
print(df.shape)

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-03-01 00:02:07 2026-03-01 00:06:24 2026-03-01 00:07:02   
1 2026-03-01 00:16:40 2026-03-01 00:20:28 2026-03-01 00:21:28   
2 2026-03-01 00:36:45 2026-03-01 00:38:16 2026-03-01 00:40:09   
3 2026-03-01 00:48:44 2026-03-01 00:51:48 2026-03-01 00:52:18   
4 2026-02-28 23:54:23 2026-02-28 23:59:48 2026-03-01 00:01:47   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  \
0 2026-03-01 00:12:16            90           249        0.52        314   
1 2026-03-01 00:31:52           113           211        1.16        625   
2 2026-03-01 00:48:45           211           113        1.25        516   
3 2026-03-01 01:07:45           114           170        2.38        927   
4 2026-03-01 00:11:19           173            70        1.01        573   

   tolls  tips  ...  time_from_request_to_pickup day_of_week hour_bucket  \
0    0.0  0.00  ...                        257.0      Sunday           0   


In [9]:
# Check for missing values after the merge
print("\nMissing values after merge:")
print(df.isnull().sum())


Missing values after merge:
request_datetime                          0
on_scene_datetime                         0
pickup_datetime                           0
dropoff_datetime                          0
PULocationID                              0
DOLocationID                              0
trip_miles                                0
trip_time                                 0
tolls                                     0
tips                                      0
driver_pay                                0
PUBorough                                 0
PUZone                                    0
DOBorough                                 0
DOZone                                    0
time_from_request_to_pickup               0
day_of_week                               0
hour_bucket                               0
average_PU_to_DO_time                     0
pickup_date                               0
PU_zone_opportunity_score                65
PU_tip_ratio_per_trip                    65
PU_

In [10]:
import pandas as pd

missing_tip_ratio_rows = df[df['PU_tip_ratio_per_trip'].isnull()]
print("Rows with missing PU_tip_ratio_per_trip:")

# Display all columns for the first 10 rows
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(missing_tip_ratio_rows.head(10))

Rows with missing PU_tip_ratio_per_trip:
           request_datetime   on_scene_datetime     pickup_datetime    dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  tolls  tips  driver_pay      PUBorough                            PUZone      DOBorough                           DOZone  time_from_request_to_pickup day_of_week  hour_bucket  average_PU_to_DO_time pickup_date  PU_zone_opportunity_score  PU_tip_ratio_per_trip  PU_avg_earnings_per_trip  PU_trip_density_per_hour  PU_avg_market_total_earnings_per_hour
65482   2026-03-01 04:12:40 2026-03-01 04:18:16 2026-03-01 04:18:16 2026-03-01 04:22:20           154           155        1.37        244    0.0   0.0        6.44       Brooklyn   Marine Park/Floyd Bennett Field       Brooklyn           Marine Park/Mill Basin                        336.0      Sunday            4             244.000000  2026-03-01                        NaN                    NaN                       NaN                       NaN                 

In [11]:
columns_with_missing = [
    'PU_zone_opportunity_score',
    'PU_tip_ratio_per_trip',
    'PU_avg_earnings_per_trip',
    'PU_trip_density_per_hour',
    'PU_avg_market_total_earnings_per_hour'
]

original_rows = df.shape[0]
df.dropna(subset=columns_with_missing, inplace=True)

print(f"Dropped {original_rows - df.shape[0]} rows with missing values.")
print(f"New DataFrame shape: {df.shape}")
print("Remaining missing values after dropping:")
print(df[columns_with_missing].isnull().sum())

Dropped 65 rows with missing values.
New DataFrame shape: (12713931, 25)
Remaining missing values after dropping:
PU_zone_opportunity_score                0
PU_tip_ratio_per_trip                    0
PU_avg_earnings_per_trip                 0
PU_trip_density_per_hour                 0
PU_avg_market_total_earnings_per_hour    0
dtype: int64


In [12]:
columns_to_merge_from_grouped_earnings = [
    'PUZone',
    'day_of_week',
    'hour_bucket',
    'PU_zone_opportunity_score',
    'PU_tip_ratio_per_trip',
    'PU_avg_earnings_per_trip',
    'PU_trip_density_per_hour',
    'PU_avg_market_total_earnings_per_hour'
]

# Create a temporary DataFrame for merging DOZone specific features
do_zone_metrics = grouped_earnings[columns_to_merge_from_grouped_earnings].copy()

# Rename PUZone to DOZone for merging on drop-off zone
do_zone_metrics.rename(columns={'PUZone': 'DOZone'}, inplace=True)

# Rename PU_ prefixed columns to DO_ prefixed columns
do_zone_metrics.rename(columns={
    'PU_zone_opportunity_score': 'DO_zone_opportunity_score',
    'PU_tip_ratio_per_trip': 'DO_tip_ratio_per_trip',
    'PU_avg_earnings_per_trip': 'DO_avg_earnings_per_trip',
    'PU_trip_density_per_hour': 'DO_trip_density_per_hour',
    'PU_avg_market_total_earnings_per_hour': 'DO_avg_market_total_earnings_per_hour'
}, inplace=True)

# Merge these new DOZone specific features into the main DataFrame
df = pd.merge(
    df,
    do_zone_metrics,
    on=['DOZone', 'day_of_week', 'hour_bucket'],
    how='left'
)

# Calculate the opportunity cost of relocation time
# The opportunity cost is the earnings lost in the PUZone during the relocation time.
# average_PU_to_DO_time is in seconds, so convert to hours by dividing by 3600.
relocation_opportunity_cost = df['PU_avg_market_total_earnings_per_hour'] * (df['average_PU_to_DO_time'] / 3600)

# Convert the relocation opportunity cost to 'opportunity score index units'
converted_relocation_opportunity_cost = relocation_opportunity_cost * conversion_factor
df['converted_relocation_opportunity_cost'] = converted_relocation_opportunity_cost

# Adjust the DO_zone_opportunity_score by subtracting the converted relocation opportunity cost
df['DO_zone_opportunity_score'] = df['DO_zone_opportunity_score'] - df['converted_relocation_opportunity_cost']

print(df[['PUZone', 'DOZone', 'day_of_week', 'hour_bucket', 'average_PU_to_DO_time', 'PU_avg_market_total_earnings_per_hour', 'converted_relocation_opportunity_cost', 'PU_zone_opportunity_score','DO_zone_opportunity_score']].head())
print(f"DataFrame shape after conversion: {df.shape}")

print(df.head())
print(df.shape)

# Check for missing values after the merge and adjustment
print("\nMissing values after merge and adjustment:")
print(df.isnull().sum())

                    PUZone                   DOZone day_of_week  hour_bucket  \
0                 Flatiron             West Village      Sunday            0   
1  Greenwich Village North                     SoHo      Sunday            0   
2                     SoHo  Greenwich Village North      Sunday            0   
3  Greenwich Village South              Murray Hill      Sunday            0   
4             North Corona            East Elmhurst    Saturday           23   

   average_PU_to_DO_time  PU_avg_market_total_earnings_per_hour  \
0             530.379310                            2372.929541   
1             485.142857                            1731.274856   
2             442.764706                            2524.268297   
3             904.878049                            3116.099071   
4             475.250000                             602.395748   

   converted_relocation_opportunity_cost  PU_zone_opportunity_score  \
0                             791.492284     

**Reasoning**:
The subtask requires converting the `relocation_opportunity_cost` into 'opportunity score index units' using the previously calculated `conversion_factor`. This step performs that multiplication.



In [13]:
import pandas as pd

# Identify columns with missing 'DO_' values (from previous output, they all have the same count)
missing_do_cols = [
    'DO_zone_opportunity_score',
    'DO_tip_ratio_per_trip',
    'DO_avg_earnings_per_trip',
    'DO_trip_density_per_hour',
    'DO_avg_market_total_earnings_per_hour'
]

# Filter DataFrame to get rows where 'DO_zone_opportunity_score' is null (implies others are also null)
missing_do_rows = df[df['DO_zone_opportunity_score'].isnull()]

print(f"Displaying 10 random rows where 'DO_' prefixed columns are null:")

# Display all columns for the 10 random rows
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(missing_do_rows.sample(10))

Displaying 10 random rows where 'DO_' prefixed columns are null:
            request_datetime   on_scene_datetime     pickup_datetime    dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  tolls   tips  driver_pay  PUBorough                         PUZone DOBorough             DOZone  time_from_request_to_pickup day_of_week  hour_bucket  average_PU_to_DO_time pickup_date  PU_zone_opportunity_score  PU_tip_ratio_per_trip  PU_avg_earnings_per_trip  PU_trip_density_per_hour  PU_avg_market_total_earnings_per_hour  DO_zone_opportunity_score  DO_tip_ratio_per_trip  DO_avg_earnings_per_trip  DO_trip_density_per_hour  DO_avg_market_total_earnings_per_hour  converted_relocation_opportunity_cost
12379093 2026-03-31 05:36:38 2026-03-31 05:37:02 2026-03-31 05:37:29 2026-03-31 06:01:48           161           138        7.73       1459   0.00   8.84       26.48  Manhattan                 Midtown Center    Queens  LaGuardia Airport                         24.0     Tuesday           

### Handle Missing DO Zone Metrics

Given that the missing values in the `DO_` prefixed columns primarily correspond to airport zones where we do not want to recommend relocation, we will fill these `NaN` values with `0`. This effectively assigns a zero opportunity score to these drop-off zones, ensuring they are not considered favorable for a driver's next pickup.

In [14]:
# Fill missing values in DO_ prefixed columns with 0
df[missing_do_cols] = df[missing_do_cols].fillna(0)

print("Missing values after filling DO_ prefixed columns with 0:")
print(df[missing_do_cols].isnull().sum())

# Verify a few random rows to ensure the fill operation was successful
print("\n5 random rows from DO_zone_opportunity_score column after filling NaN with 0:")
print(df[['DOZone', 'day_of_week', 'hour_bucket', 'DO_zone_opportunity_score']].sample(5))

Missing values after filling DO_ prefixed columns with 0:
DO_zone_opportunity_score                0
DO_tip_ratio_per_trip                    0
DO_avg_earnings_per_trip                 0
DO_trip_density_per_hour                 0
DO_avg_market_total_earnings_per_hour    0
dtype: int64

5 random rows from DO_zone_opportunity_score column after filling NaN with 0:
                 DOZone day_of_week  hour_bucket  DO_zone_opportunity_score
11982119   Clinton East      Sunday           23                3588.917330
5382947   Melrose South    Saturday            9                3720.777027
58699      Clinton Hill      Sunday            2                2950.427167
8490676     Brownsville    Saturday           18                5815.134638
4474833    East Tremont    Thursday            8                6477.910657


In [15]:
# Columns to display for a more comprehensive comparison
display_columns = [
    'PUZone',
    'DOZone',
    'day_of_week',
    'hour_bucket',
    'average_PU_to_DO_time',
    'PU_zone_opportunity_score',
    'DO_zone_opportunity_score',
    'PU_avg_market_total_earnings_per_hour',
    'DO_avg_market_total_earnings_per_hour',
    'PU_tip_ratio_per_trip',
    'DO_tip_ratio_per_trip'
]

# 5 random rows where DO opportunity score was higher than PU opportunity score
higher_do_score = df[df['DO_zone_opportunity_score'] > df['PU_zone_opportunity_score']]
print(f"\nNumber of rows where DO opportunity score is HIGHER than PU opportunity score: {higher_do_score.shape[0]}")
print("5 random rows where DO opportunity score is HIGHER than PU opportunity score:")
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(higher_do_score.sample(5)[display_columns])

# 5 random rows where DO opportunity score was lower than PU opportunity score
lower_do_score = df[df['DO_zone_opportunity_score'] < df['PU_zone_opportunity_score']]
print(f"\nNumber of rows where DO opportunity score is LOWER than PU opportunity score: {lower_do_score.shape[0]}")
print("5 random rows where DO opportunity score is LOWER than PU opportunity score:")
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(lower_do_score.sample(5)[display_columns])

# 5 random rows where DO opportunity score was the same as PU opportunity score
same_do_score = df[df['DO_zone_opportunity_score'] == df['PU_zone_opportunity_score']]
print(f"\nNumber of rows where DO opportunity score is the SAME as PU opportunity score: {same_do_score.shape[0]}")
print("5 random rows where DO opportunity score is the SAME as PU opportunity score:")
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(same_do_score.sample(5)[display_columns])


Number of rows where DO opportunity score is HIGHER than PU opportunity score: 4442035
5 random rows where DO opportunity score is HIGHER than PU opportunity score:
                                  PUZone          DOZone day_of_week  hour_bucket  average_PU_to_DO_time  PU_zone_opportunity_score  DO_zone_opportunity_score  PU_avg_market_total_earnings_per_hour  DO_avg_market_total_earnings_per_hour  PU_tip_ratio_per_trip  DO_tip_ratio_per_trip
10543961                        Canarsie   East New York    Thursday           20             728.948718                5292.948152                9874.724721                            1181.432870                            2115.653091               0.016624               0.010483
2674598              Little Italy/NoLiTa        Union Sq    Saturday           13             968.105263                8163.707163                9709.504774                            1829.574434                            2264.462406               0.068168         

## Step 3

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12713931 entries, 0 to 12713930
Data columns (total 31 columns):
 #   Column                                 Dtype         
---  ------                                 -----         
 0   request_datetime                       datetime64[us]
 1   on_scene_datetime                      datetime64[us]
 2   pickup_datetime                        datetime64[us]
 3   dropoff_datetime                       datetime64[us]
 4   PULocationID                           int32         
 5   DOLocationID                           int32         
 6   trip_miles                             float64       
 7   trip_time                              int64         
 8   tolls                                  float64       
 9   tips                                   float64       
 10  driver_pay                             float64       
 11  PUBorough                              object        
 12  PUZone                                 object        


In [17]:
df['day_of_week_numeric'] = df['request_datetime'].dt.dayofweek

columns_to_drop = [
    'request_datetime',
    'on_scene_datetime',
    'pickup_datetime',
    'dropoff_datetime',
    'pickup_date',
    'converted_relocation_opportunity_cost',
    'day_of_week' # Add the original object-type day_of_week to be dropped
]

df = df.drop(columns=columns_to_drop)

print("DataFrame after dropping columns and converting day_of_week:")
print(df.info())
df.head()

DataFrame after dropping columns and converting day_of_week:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12713931 entries, 0 to 12713930
Data columns (total 25 columns):
 #   Column                                 Dtype  
---  ------                                 -----  
 0   PULocationID                           int32  
 1   DOLocationID                           int32  
 2   trip_miles                             float64
 3   trip_time                              int64  
 4   tolls                                  float64
 5   tips                                   float64
 6   driver_pay                             float64
 7   PUBorough                              object 
 8   PUZone                                 object 
 9   DOBorough                              object 
 10  DOZone                                 object 
 11  time_from_request_to_pickup            float64
 12  hour_bucket                            int32  
 13  average_PU_to_DO_time                  

,PULocationID,DOLocationID,trip_miles,trip_time,tolls,tips,driver_pay,PUBorough,PUZone,DOBorough,...,PU_tip_ratio_per_trip,PU_avg_earnings_per_trip,PU_trip_density_per_hour,PU_avg_market_total_earnings_per_hour,DO_zone_opportunity_score,DO_tip_ratio_per_trip,DO_avg_earnings_per_trip,DO_trip_density_per_hour,DO_avg_market_total_earnings_per_hour,day_of_week_numeric
0,90,249,0.52,314,0.0,0.00,19.30,Manhattan,Flatiron,Manhattan,...,0.031589,23.200932,101.963309,2372.929541,18292.464667,0.040196,26.464698,163.009298,3742.008749,6
1,113,211,1.16,625,0.0,0.00,23.36,Manhattan,Greenwich Village North,Manhattan,...,0.036698,26.865305,64.423936,1731.274856,11014.324448,0.039839,26.142864,95.798154,2524.268297,6
2,211,113,1.25,516,0.0,9.33,26.19,Manhattan,SoHo,Manhattan,...,0.039839,26.142864,95.798154,2524.268297,6535.382393,0.036698,26.865305,64.423936,1731.274856,6
3,114,170,2.38,927,0.0,5.35,22.19,Manhattan,Greenwich Village South,Manhattan,...,0.042498,27.028737,134.191086,3116.099071,6888.118814,0.046227,20.543218,92.788593,1907.358142,6
4,173,70,1.01,573,0.0,0.00,7.99,Queens,North Corona,Queens,...,0.021131,14.540916,45.556414,602.395748,1081.424854,0.025194,18.128750,24.347935,390.221853,5


In [18]:
agg_df = (
    df.groupby([
        'PULocationID',
        'DOLocationID',
        'hour_bucket',
        'day_of_week_numeric'
    ])
    .agg({
        'average_PU_to_DO_time': 'mean',
        'PU_avg_market_total_earnings_per_hour': 'mean',
        'DO_avg_market_total_earnings_per_hour': 'mean',
        'PU_trip_density_per_hour': 'mean',
        'DO_trip_density_per_hour': 'mean',
        'PU_zone_opportunity_score': 'mean',
        'DO_zone_opportunity_score': 'mean'
    })
    .reset_index()
)

print("Aggregated DataFrame Head:")
display(agg_df.head())

Aggregated DataFrame Head:


,PULocationID,DOLocationID,hour_bucket,day_of_week_numeric,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score
0,2,39,17,0,1333.0,67.754914,1387.954591,1.600000,98.647725,64.740596,6330.069330
1,2,39,17,4,1624.0,65.394089,1426.242904,2.216749,94.111871,76.404567,6429.824941
2,2,39,18,1,1318.0,94.796439,1441.177206,4.272997,105.094787,157.608455,6643.555575
3,2,39,18,2,1343.0,68.008929,1373.524527,2.678571,99.529997,88.583294,6275.134254
4,2,45,16,5,4175.0,56.841198,889.556143,0.862275,34.848010,35.343806,3034.734338


In [19]:
agg_df['travel_penalty'] = (
    (agg_df['average_PU_to_DO_time'] / 3600) # Convert seconds to hours
    * agg_df['PU_avg_market_total_earnings_per_hour']
)

print("Aggregated DataFrame with Travel Penalty Head:")
display(agg_df.head())

Aggregated DataFrame with Travel Penalty Head:


,PULocationID,DOLocationID,hour_bucket,day_of_week_numeric,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score,travel_penalty
0,2,39,17,0,1333.0,67.754914,1387.954591,1.600000,98.647725,64.740596,6330.069330,25.088139
1,2,39,17,4,1624.0,65.394089,1426.242904,2.216749,94.111871,76.404567,6429.824941,29.500000
2,2,39,18,1,1318.0,94.796439,1441.177206,4.272997,105.094787,157.608455,6643.555575,34.706030
3,2,39,18,2,1343.0,68.008929,1373.524527,2.678571,99.529997,88.583294,6275.134254,25.371109
4,2,45,16,5,4175.0,56.841198,889.556143,0.862275,34.848010,35.343806,3034.734338,65.920000


In [20]:
agg_df['net_gain'] = (
    agg_df['DO_avg_market_total_earnings_per_hour']
    - agg_df['PU_avg_market_total_earnings_per_hour']
    - agg_df['travel_penalty']
)

print("Aggregated DataFrame with Net Gain Head:")
display(agg_df.head())

Aggregated DataFrame with Net Gain Head:


,PULocationID,DOLocationID,hour_bucket,day_of_week_numeric,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score,travel_penalty,net_gain
0,2,39,17,0,1333.0,67.754914,1387.954591,1.600000,98.647725,64.740596,6330.069330,25.088139,1295.111538
1,2,39,17,4,1624.0,65.394089,1426.242904,2.216749,94.111871,76.404567,6429.824941,29.500000,1331.348815
2,2,39,18,1,1318.0,94.796439,1441.177206,4.272997,105.094787,157.608455,6643.555575,34.706030,1311.674737
3,2,39,18,2,1343.0,68.008929,1373.524527,2.678571,99.529997,88.583294,6275.134254,25.371109,1280.144490
4,2,45,16,5,4175.0,56.841198,889.556143,0.862275,34.848010,35.343806,3034.734338,65.920000,766.794946


In [21]:
stay_rows = (
    agg_df.groupby([
        'PULocationID',
        'hour_bucket',
        'day_of_week_numeric'
    ])
    .first() # Take the first occurrence for each group as a base
    .reset_index()
)

stay_rows['DOLocationID'] = stay_rows['PULocationID']
stay_rows['average_PU_to_DO_time'] = 0
stay_rows['travel_penalty'] = 0

stay_rows['DO_avg_market_total_earnings_per_hour'] = (
    stay_rows['PU_avg_market_total_earnings_per_hour']
)

stay_rows['net_gain'] = 0

print("Stay-Put Rows Head:")
display(stay_rows.head())

Stay-Put Rows Head:


,PULocationID,hour_bucket,day_of_week_numeric,DOLocationID,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score,travel_penalty,net_gain
0,2,3,2,2,0,68.354430,68.354430,5.696203,5.280194,129.978723,216.363895,0,0
1,2,7,4,2,0,60.645807,60.645807,4.505632,7.776807,103.447894,496.726519,0,0
2,2,9,6,2,0,65.353371,65.353371,1.545728,111.949070,61.067273,8948.046553,0,0
3,2,11,2,2,0,66.279294,66.279294,5.778491,24.553766,126.842299,1138.540587,0,0
4,2,11,4,2,0,53.216343,53.216343,1.607502,19.869080,51.002147,1128.486149,0,0


In [22]:
training_table = pd.concat([
    agg_df,
    stay_rows
], ignore_index=True)

print("Final Training Table Head:")
display(training_table.head())

Final Training Table Head:


,PULocationID,DOLocationID,hour_bucket,day_of_week_numeric,average_PU_to_DO_time,PU_avg_market_total_earnings_per_hour,DO_avg_market_total_earnings_per_hour,PU_trip_density_per_hour,DO_trip_density_per_hour,PU_zone_opportunity_score,DO_zone_opportunity_score,travel_penalty,net_gain
0,2,39,17,0,1333.0,67.754914,1387.954591,1.600000,98.647725,64.740596,6330.069330,25.088139,1295.111538
1,2,39,17,4,1624.0,65.394089,1426.242904,2.216749,94.111871,76.404567,6429.824941,29.500000,1331.348815
2,2,39,18,1,1318.0,94.796439,1441.177206,4.272997,105.094787,157.608455,6643.555575,34.706030,1311.674737
3,2,39,18,2,1343.0,68.008929,1373.524527,2.678571,99.529997,88.583294,6275.134254,25.371109,1280.144490
4,2,45,16,5,4175.0,56.841198,889.556143,0.862275,34.848010,35.343806,3034.734338,65.920000,766.794946


In [23]:
training_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2548903 entries, 0 to 2548902
Data columns (total 13 columns):
 #   Column                                 Dtype  
---  ------                                 -----  
 0   PULocationID                           int32  
 1   DOLocationID                           int32  
 2   hour_bucket                            int32  
 3   day_of_week_numeric                    int32  
 4   average_PU_to_DO_time                  float64
 5   PU_avg_market_total_earnings_per_hour  float64
 6   DO_avg_market_total_earnings_per_hour  float64
 7   PU_trip_density_per_hour               float64
 8   DO_trip_density_per_hour               float64
 9   PU_zone_opportunity_score              float64
 10  DO_zone_opportunity_score              float64
 11  travel_penalty                         float64
 12  net_gain                               float64
dtypes: float64(9), int32(4)
memory usage: 213.9 MB


## Exporting DataFrames

In [24]:
# Export the main processed training_table 'training_table' to Parquet
output_test_table_path = '/content/uber_trips_test.parquet'
training_table.to_parquet(output_test_table_path, index=False)
print(f"Main DataFrame exported to: {output_test_table_path}")

Main DataFrame exported to: /content/uber_trips_test.parquet


### Export for Zone to Zone Lookups

In [25]:
# # Export 'average_trip_time_by_route_time' for zone-to-zone lookups
# output_avg_trip_time_path = '/content/avg_trip_time_zone_to_zone.parquet'
# average_trip_time_by_route_time.to_parquet(output_avg_trip_time_path, index=False)
# print(f"Average trip time DataFrame exported to: {output_avg_trip_time_path}")

### Export for Earnings Estimations

In [26]:
# # Export 'grouped_earnings' for earnings estimations
# output_grouped_earnings_path = '/content/avg_earnings_zone_hour.parquet'
# grouped_earnings.to_parquet(output_grouped_earnings_path, index=False)
# print(f"Grouped earnings DataFrame exported to: {output_grouped_earnings_path}")